# 🔧 Tool Calling Hands-on Lab

What you'll build:
- Understand **how tool calling works** — without any API key
- Define Python functions as **tools** an LLM can call
- Use **Gemini API** (free, your own key) to let the model decide which tool to call
- See the model call **multiple tools** in one conversation

**Phase 1 is completely free** — no API key required.
**Phases 2–4 require a free Gemini API key** — get one at [aistudio.google.com](https://aistudio.google.com).

Run each cell top to bottom with **Shift+Enter**.

---

## Phase 1: Understand the Mechanism (No API Key)
### Cell 1 — Define 3 tools as Python functions

Tools are just regular Python functions. The LLM reads the **name** and **docstring** to decide when to call each one.

In [ ]:
from datetime import date

# ── Tool 1: Math calculator ──────────────────────────────────────────
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '240 * 0.15', '(100 + 50) / 2'"""
    try:
        allowed = {"abs": abs, "round": round, "max": max, "min": min, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# ── Tool 2: Current date ─────────────────────────────────────────────
def get_today() -> str:
    """Get today's date in YYYY-MM-DD format."""
    return str(date.today())

# ── Tool 3: SE knowledge search ──────────────────────────────────────
def search_docs(query: str) -> str:
    """Search software engineering documentation. Use for technical questions about Git, JWT, Docker, REST, etc."""
    knowledge = {
        "git":         "Git is a distributed version control system. Tracks changes with commits and branches.",
        "jwt":         "JWT (JSON Web Token) has Header, Payload, Signature. Tokens expire after 24 hours by default.",
        "docker":      "Docker packages apps as containers using a Dockerfile. Solves 'works on my machine' problem.",
        "rest":        "REST API uses HTTP: GET (read), POST (create), PUT (update), DELETE. Stateless design.",
        "cache":       "Caching stores frequent data in fast memory. Redis/Memcached are popular. TTL sets expiration.",
        "cicd":        "CI/CD automates tests, builds, and deploys on code push. Tools: GitHub Actions, Jenkins.",
        "database":    "Database index speeds up queries using B-Tree. Without index: Full Table Scan.",
        "microservice":"Microservices: small independent services, each deployable separately via REST or message queues.",
        "http":        "HTTP status codes: 200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 404 Not Found, 500 Error.",
        "test":        "Testing: unit, integration, E2E. TDD writes tests first. Frameworks: Jest, JUnit, PyTest.",
    }
    query_lower = query.lower()
    results = []
    for key, content in knowledge.items():
        if key in query_lower or any(w in content.lower() for w in query_lower.split() if len(w) > 3):
            results.append(f"[{key}] {content}")
    return "\n\n".join(results[:2]) if results else f"No results found for: {query}"

# Tool registry — maps name → function
tool_map = {
    "calculate":   calculate,
    "get_today":   get_today,
    "search_docs": search_docs,
}

print("✅ 3 tools defined")
print(f"   calculate   → {calculate.__doc__.split('.')[0]}")
print(f"   get_today   → {get_today.__doc__}")
print(f"   search_docs → {search_docs.__doc__.split('.')[0]}")

### Cell 2 — What a tool call looks like

When the LLM decides to use a tool, it outputs a JSON object — **not** the answer itself.
Your code reads that JSON, runs the function, and sends the result back.

In [ ]:
import json

# This is what the LLM outputs when it wants to call a tool:
llm_tool_call = {
    "type": "tool_use",
    "name": "calculate",
    "args": { "expression": "240 * 0.15" }
}

print("LLM output (instead of answering directly):")
print(json.dumps(llm_tool_call, indent=2))
print()
print("→ LLM is saying: 'I want to call calculate(expression=\"240 * 0.15\")'")
print("→ Your code executes it. The LLM waits for the result.")

### Cell 3 — Execute a tool call

Read the JSON → look up the function → call it → get the result.

In [ ]:
# Step 1: LLM outputs this (simulated)
llm_output = {
    "name": "calculate",
    "args": { "expression": "240 * 0.15" }
}

# Step 2: Your code executes it
fn_name = llm_output["name"]
fn_args = llm_output["args"]
result  = tool_map[fn_name](**fn_args)  # calls calculate(expression="240 * 0.15")

print(f"Tool called : {fn_name}")
print(f"Arguments  : {fn_args}")
print(f"Result     : {result}")
print()

# Step 3: Inject result back → LLM generates final answer
injected_context = f"Tool result for {fn_name}: {result}"
print(f"Injected into context: '{injected_context}'")
print("→ LLM now answers: '15% of 240 is 36.'")

### Cell 4 — Simulate a multi-tool loop

A real question often needs **more than one tool**. The LLM keeps calling tools until it has everything it needs.

In [ ]:
print('User: "What is JWT? And how many days is 3 weeks?"')
print("=" * 60)

# Simulated: LLM decides to call 2 tools
simulated_calls = [
    {"name": "search_docs", "args": {"query": "JWT"}},
    {"name": "calculate",   "args": {"expression": "3 * 7"}},
]

results = []
for i, call in enumerate(simulated_calls, 1):
    result = tool_map[call["name"]](**call["args"])
    results.append(result)
    print(f"\nTurn {i}: LLM calls → {call['name']}({call['args']})")
    print(f"         Result   → {result[:80]}..." if len(result) > 80 else f"         Result   → {result}")

print("\n" + "=" * 60)
print("Turn 3: LLM has all context → generates final answer:")
print(f"  JWT tokens expire after 24 hours (Header + Payload + Signature).")
print(f"  3 weeks = {results[1]} days.")
print()
print("This loop is exactly what the while loop in the code does automatically.")

---
## Phase 2: Real Tool Calling with Gemini (API Key Required)
### Cell 5 — Install and set up API key

Get your free API key at **[aistudio.google.com](https://aistudio.google.com)** → Sign in with Google → Get API key.

In [ ]:
!pip install -q google-generativeai
print("✅ google-generativeai installed")

In [ ]:
import google.generativeai as genai

# ✏️ Paste your Gemini API key here
GEMINI_API_KEY = "AIza..."  # replace with your key

genai.configure(api_key=GEMINI_API_KEY)
print("✅ Gemini configured")

# ❌ Never print your key:
# print(GEMINI_API_KEY)

### Cell 6 — Single tool: let Gemini call calculate

Pass the function to the model. Gemini reads the docstring to understand what it does.

In [ ]:
# Give Gemini one tool
model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    tools=[calculate]   # pass the Python function directly
)
chat = model.start_chat(enable_automatic_function_calling=True)

# ✏️ Try changing this question
question = "What is 15% of 240? Also what is 300 divided by 8?"

response = chat.send_message(question)

print(f"Q: {question}")
print(f"\nA: {response.text}")

### Cell 7 — Multiple tools: Gemini picks the right one

Give Gemini all 3 tools. Ask a question that requires **different tools** — watch it decide which to call.

In [ ]:
# Give Gemini all 3 tools
model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    tools=[calculate, get_today, search_docs]
)
chat = model.start_chat(enable_automatic_function_calling=True)

# ✏️ Try changing this question
question = "What is JWT and how long does a token stay valid? Also what's today's date?"

print(f"Q: {question}\n")
response = chat.send_message(question)
print(f"A: {response.text}")

### Cell 8 — See which tools were called

Look inside the conversation history to see **exactly** which tools Gemini decided to use.

In [ ]:
# Inspect the conversation history
print("Tool calls made in this conversation:")
print("=" * 50)

for content in chat.history:
    for part in content.parts:
        if hasattr(part, 'function_call') and part.function_call.name:
            fn = part.function_call
            print(f"\n🔧 Called : {fn.name}")
            print(f"   Args   : {dict(fn.args)}")
        elif hasattr(part, 'function_response') and part.function_response.name:
            fr = part.function_response
            result_str = str(fr.response)[:100]
            print(f"   Result : {result_str}..." if len(str(fr.response)) > 100 else f"   Result : {fr.response}")

### Cell 9 — ✏️ Ask your own question

Try questions that combine multiple tools. Some examples:
- `"How does Docker work? If I have 3 containers each using 512MB RAM, how much total?"`
- `"What is CI/CD? If a deploy runs every 2 days, how many deploys in 3 weeks?"`
- `"What is caching? What's today's date and how many days until the end of the year?"`

In [ ]:
model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    tools=[calculate, get_today, search_docs]
)
chat = model.start_chat(enable_automatic_function_calling=True)

# ✏️ Write your own question here
my_question = "How does Docker work? If I have 3 containers each using 512MB, how much total RAM?"

print(f"Q: {my_question}\n")
response = chat.send_message(my_question)
print(f"A: {response.text}")

---
## 🎉 Done!

What you built:

| Step | What happened |
|------|---------------|
| Define tools | Python functions with docstrings |
| Tool call JSON | LLM outputs `{name, args}` instead of an answer |
| Execute | Your code runs the function, returns result |
| Inject | Result goes back into context |
| LLM answers | Final response using real data |

**Key insight:** The LLM never runs code. It outputs *descriptions* of what to run. You run it. You return the result. This is why tool calling is safe — you control what tools exist and what they can do.

**This is how GitHub Copilot reads your files, Cursor runs your tests, and Claude Code edits your code.**